# 1. Triage & Alert Correlation

Welcome to the SOC. A real analyst's day doesn't start with "hunt for evil" — it starts with **a queue of alerts**, and the first skill you need is deciding **what to look at first**.

This notebook teaches:

1. How raw alerts get grouped into **incidents** by shared entities (the "correlate" step).
2. How to **triage** an incident queue: not just severity, but recency, confidence, and blast radius.
3. A bad → best progression you can use on the exam AND in a real SOC.

> **SC-200 mapping**: "Triage security alerts and incidents" and "Manage incidents in Microsoft Defender XDR and Microsoft Sentinel".


## Setup

This lab reuses the mini-SIEM from Lab 1. Make sure it's running:

```bash
cd ../../01-build-a-siem && docker compose up -d
```

Then, in VS Code:
1. Pick the **`.venv` kernel** from this folder (top-right kernel picker).
2. If it's missing, reload the window (`Cmd+Shift+P` → `Reload Window`).

All cells talk to `http://localhost:8000` — the same mini-SIEM container seeded with a realistic multi-stage attack plus normal background traffic.


In [ ]:
import httpx
from collections import Counter, defaultdict
from datetime import datetime

SIEM = 'http://localhost:8000'

# Sanity check: can we reach the SIEM?
health = httpx.get(f'{SIEM}/health').json()
print('SIEM health:', health)

dashboard = httpx.get(f'{SIEM}/dashboard').json()
print('Dashboard:', dashboard)


## Step 1 — Look at raw alerts

Each **alert** is a single detection firing once. The mini-SIEM exposes them at `GET /alerts`.

Think of alerts as tickets an analyst could be asked to read. There are usually dozens per hour in a real tenant.


In [ ]:
alerts = httpx.get(f'{SIEM}/alerts').json()
print(f'Total alerts: {len(alerts)}\n')

for a in alerts[:10]:
    print(f"[{a['severity']:<8}] {a['rule_name']:<32} tactic={a['tactic']!s:<18} status={a['status']}")
    print(f"           {a['title']}")


## Step 2 — Correlate alerts into incidents

Several alerts can be **the same story**: a brute-force alert and a "sign-in from suspicious location" alert for the same user are almost certainly one incident, not two.

In Defender XDR and Sentinel, the platform groups them automatically. Our mini-SIEM exposes the same idea at `POST /incidents/correlate` — it groups "New" alerts by shared entities (e.g. `UserPrincipalName`).

The call is **idempotent-ish**: once an alert's status moves to `InIncident`, re-running won't create a duplicate incident for it.


In [ ]:
correlated = httpx.post(f'{SIEM}/incidents/correlate').json()
print('Correlation result:', correlated)

incidents = httpx.get(f'{SIEM}/incidents').json()
print(f'\nTotal incidents now: {len(incidents)}')
for inc in incidents:
    print(f"  {inc['id']}  sev={inc['severity']:<8} status={inc['status']:<6}  {inc['title']}")


## Step 3 — Bad triage: first-in-first-out

The **bad** approach is to just walk the queue top to bottom. You will burn hours on low-impact alerts while a real breach sits unanswered.


In [ ]:
print('❌ BAD: first-in-first-out (no prioritization)')
for inc in incidents:
    print(f"  → work on {inc['id']}  ({inc['severity']})")


## Step 4 — Better triage: severity + recency

A small step up is to sort by severity, tie-break by recency. This is what most junior SOCs actually do.


In [ ]:
SEV_ORDER = {'Critical': 4, 'High': 3, 'Medium': 2, 'Low': 1, 'Informational': 0}

better = sorted(
    incidents,
    key=lambda i: (SEV_ORDER.get(i['severity'], 0), i['created_at']),
    reverse=True,
)
print('⚠️  BETTER: severity then recency')
for inc in better:
    print(f"  {inc['severity']:<8} {inc['created_at']}  {inc['id']}  {inc['title']}")


## Step 5 — Best triage: severity + confidence + blast radius + containment

Real SOC triage weighs four things, not one:

| Factor | Question |
|---|---|
| **Severity** | How bad is it if this is real? |
| **Confidence** | How likely is the detection real vs a false positive? Multiple correlated alerts ⇒ higher confidence. |
| **Blast radius** | How many users/devices/services are affected? |
| **Containment state** | Has automation already contained the threat? If yes, drop the urgency. |

We'll approximate each factor from the incident data and produce a **priority score**. Higher = look at first.


In [ ]:
import json as _json

# --- Step A: pull every incident WITH its alerts, once. ---
details = {i['id']: httpx.get(f"{SIEM}/incidents/{i['id']}").json() for i in incidents}


def entities_of(detail):
    """Every (type, value) entity referenced by any alert in the incident."""
    out = set()
    for a in detail.get('alerts', []):
        for ev in _json.loads(a['evidence'] or '[]'):
            for key in ('UserPrincipalName', 'DeviceName', 'AccountName', 'SourceIP', 'IPAddress'):
                if ev.get(key):
                    out.add((key, ev[key]))
    return out


ENTITIES = {iid: entities_of(d) for iid, d in details.items()}

# --- Step B: which incidents SHARE an entity with which other incidents? ---
# This is the single most important triage signal and the one juniors miss:
# five "separate" mediums that all touch alice are one campaign, not five tickets.
def campaign_links(iid):
    mine = ENTITIES[iid]
    return {other for other, ents in ENTITIES.items() if other != iid and (mine & ents)}


def score(incident):
    iid = incident['id']
    detail = details[iid]
    alerts = detail.get('alerts', [])

    # 1. Severity — how bad if real.
    sev = SEV_ORDER.get(incident['severity'], 0)

    # 2. Confidence — independent corroboration. Two sources, not one loud rule.
    #    (a) alerts inside this incident, (b) OTHER incidents sharing an entity.
    linked = campaign_links(iid)
    confidence = min(len(alerts) + len(linked), 5)

    # 3. Blast radius — distinct users/devices/IPs implicated.
    blast = min(len(ENTITIES[iid]), 10)

    # 4. Containment — already contained? urgency drops (investigation does not).
    containment_penalty = 0 if incident['status'] == 'New' else -2

    priority = sev * 10 + confidence * 3 + blast * 2 + containment_penalty
    return priority, {
        'severity': incident['severity'],
        'alerts': len(alerts),
        'linked': len(linked),
        'blast': blast,
        'status': incident['status'],
        'priority': priority,
    }


ranked = sorted(((score(i), i) for i in incidents), key=lambda x: x[0][0], reverse=True)

print('✅ BEST: severity + confidence (alerts + campaign links) + blast radius + containment\n')
print(f'{"incident":<12} {"prio":>4}  {"sev":<7} {"alerts":>6} {"linked":>7} {"blast":>6}  {"status":<7} title')
print('-' * 118)
for (p, meta), inc in ranked:
    print(f"{inc['id']:<12} {meta['priority']:>4}  {meta['severity']:<7} {meta['alerts']:>6} "
          f"{meta['linked']:>7} {meta['blast']:>6}  {meta['status']:<7} {inc['title']}")

print('\n--- Why the order changed: shared entities across incidents ---')
for iid in [i['id'] for i in incidents]:
    linked = campaign_links(iid)
    if linked:
        shared = sorted({v for other in linked for (k, v) in (ENTITIES[iid] & ENTITIES[other])})
        print(f"  {iid} shares {shared} with {sorted(linked)}")
print('\nAny incident with links is part of ONE campaign. Work the campaign, not the tickets.')


## What you just did (SC-200 mapping)

| You did... | Real portal equivalent |
|---|---|
| `GET /alerts` | Defender XDR → Alerts queue |
| `POST /incidents/correlate` | XDR's automatic alert-to-incident correlation |
| Ranked by severity + confidence + blast + containment | The **Priority** column and the analyst's mental model |
| Inspected incident with its alerts | Incident page → Alerts tab |

### Exam tips

- **Correlation is the point of an incident.** Don't work alerts one by one when they share entities.
- **Severity alone is a weak ranking.** Two High incidents with different blast radius are not equal.
- If **automatic attack disruption** already contained the threat, triage urgency drops — but investigation urgency does not.

➡️ Next: [02 — Entity-centric investigation](02_entity_pivot_investigation.ipynb)


---
## ✅ Self-check

1. Two incidents are both `High`. One touches a single test VM, the other touches 40 servers
   and a domain admin account. Why does severity alone rank them equally, and what fixes it?
2. Automatic attack disruption has already contained a device. Does the incident's **triage**
   urgency change? Does its **investigation** urgency change?
3. You have five separate Medium incidents that all reference `alice@contoso.com`. What should
   you do before working any of them?
4. What is the difference between an **alert** and an **incident**?
5. Why is "number of alerts in the incident" a proxy for *confidence* rather than for
   *severity*?

In [ ]:
answers = """
1. Severity answers only "how bad if this is real". It says nothing about BLAST RADIUS
   (how much of the estate is implicated) or CONFIDENCE (how corroborated the signal
   is). Adding those two dimensions -- as the scorer above does -- separates the test
   VM from the 40-server incident.

2. Triage urgency DROPS -- the bleeding has stopped, so you are no longer racing the
   attacker. Investigation urgency does NOT drop: you still have to establish scope,
   root cause and whether anything else was touched before you release the containment.
   Treating "contained" as "closed" is how re-entry happens.

3. Check whether they are ONE campaign. Look for shared entities (user, IP, device,
   file hash). If they share entities, merge/link them and investigate as a single
   story -- otherwise five analysts each write a partial report and nobody sees the
   kill chain. In Sentinel you can merge incidents; in Defender XDR the correlation
   engine usually does it for you, but only for signals it knows are related.

4. An ALERT is one detection firing once -- a single rule, a single condition met.
   An INCIDENT is a container that groups related alerts (plus their entities,
   comments, classification and owner) into one investigable story. You classify and
   close incidents, not alerts.

5. Because severity is a property of the DETECTION (what would it mean if true),
   while multiple independent detections firing on the same entities is evidence that
   the detections are RIGHT. A single noisy rule firing ten times is not corroboration
   -- which is why the scorer counts distinct alerts and distinct linked incidents,
   and caps the contribution.
"""
print(answers)